# Escalabilidad, Rendimiento y Costos (Senior)

Objetivos:
- Optimizar rendimiento físico/lógico en DW/Lakehouse.
- Comparar tácticas en Snowflake, BigQuery y Databricks (Delta).
- Gestionar costos (FinOps): presupuestos, tags, chargeback/showback.

## 1. Modelos de Costo (alto nivel)

- Snowflake: créditos por compute (virtual warehouses, tamaños), storage aparte; auto-suspend/resume; multi-cluster para concurrencia.
- BigQuery: compute por bytes procesados (on-demand) o slots (reservations); storage separado; particiones/cluster reducen bytes escaneados.
- Databricks: DBUs + costo cloud; clusters autoescalables; Photon (aceleración), Delta Lake (OPTIMIZE/Z-ORDER).

Buenas prácticas de FinOps:
- Etiquetado (tags/labels) por dominio/proyecto.
- Presupuestos y alertas; gobernar tamaños de clúster y horarios.
- Catálogo de costos: dashboards por dominio (chargeback/showback).

## 2. Diseño Físico y Pruning

Técnicas comunes:
- Particionamiento temporal/geográfico.
- Clustering/ordenamiento (sort keys, Z-ORDER).
- Formatos columnares (Parquet/ORC) y compresión adecuada.
- Estadísticas actualizadas y materialized views seleccionadas.
- Proyección de columnas (evitar SELECT *).

In [ ]:
# Demo conceptual: proyección de columnas y pruning con pandas
import pandas as pd, numpy as np
np.random.seed(1)
n = 200_000
df = pd.DataFrame({
  'fecha': pd.date_range('2025-01-01', periods=n, freq='min'),
  'region': np.random.choice(['Norte','Sur','Este','Oeste'], size=n),
  'sku': np.random.randint(1, 5000, size=n),
  'precio': np.random.uniform(5, 500, size=n),
  'cantidad': np.random.randint(1, 10, size=n)
})
# Calcular importe total (columna derivada)
df['importe'] = df['precio'] * df['cantidad']

# 1) SELECT * (lee todas las columnas)
sample_all = df.head(1000).copy()
# 2) Proyección (solo columnas necesarias)
sample_proj = df[['region','importe']].head(1000).copy()
print('Columnas all:', sample_all.columns.tolist())
print('Columnas proyección:', sample_proj.columns.tolist())

# 3) Pruning por filtro de partición (simulado por fecha)
mask = (df['fecha'] >= '2025-02-01') & (df['fecha'] < '2025-03-01')
febrero = df.loc[mask, ['fecha','region','importe']].head(3)
print('
Ejemplo pruning febrero (3 filas):
', febrero)

## 3. Tácticas por Plataforma

Snowflake:
- Auto-suspend/resume de warehouses; dimensionar por carga, escalar a picos.
- Result/metadata caching; clustering keys en tablas grandes.
- Multi-cluster warehouses para concurrencia; query acceleration opcional.

BigQuery:
- Tablas particionadas (por fecha o ingestion-time) y clustering (hasta 4 cols).
- Materialized views para agregaciones repetidas; BI Engine para dashboards.
- Evitar SELECT *; usar filtros de partición; control de bytes escaneados.

Databricks (Delta):
- Photon; Auto-Optimize; OPTIMIZE + ZORDER(col1,col2) para locality.
- Caching y AQE (Adaptive Query Execution) en Spark.
- Autoscaling y políticas de clúster; spot/low-priority cuando aplique.

## 4. Workload Management y Concurrencia

- Separar workloads: ingestión, transformaciones, BI/serving (compute aislado).
- Programación inteligente (ventanas nocturnas, micro-batch).
- Pooling de recursos y límites de cuotas; SLAs/SLOs por cola.

## 5. Cost Patterns

- Auto-suspend en Snowflake; reservations/slots adecuados en BigQuery.
- Cerrar clústeres o escalar a cero en Databricks fuera de horario.
- Seleccionar tamaños de archivo óptimos (128–512 MB) para Parquet.
- Catálogo de costos con tags/labels + reportes periódicos.

## 6. Ejercicio

Escenario: dashboard con latencia P95 < 2s (30 días de datos).
- Propón partición/clustering por plataforma.
- Define compute (tamaño warehouse/slots/cluster) y horarios.
- Lista 5 tácticas de reducción de costo sin degradar SLO.

In [ ]:
plantilla = '''
# Solución (borrador)
Partición/Clustering: [Snowflake/BigQuery/Databricks]
Compute: [WH size / slots / cluster size]
Tácticas costo: [lista]
Monitoreo: [métricas de costo y rendimiento]
'''
print(plantilla)

## 7. Resumen

- Diseña para pruning y proyección; evita SELECT *.
- Ajusta compute a patrones reales y automatiza suspensión.
- Mide costo y rendimiento con métricas y alertas.